# Render Remotion 1080p 30fps di Google Colab (T4 GPU)

Sebelum menjalankan sel di bawah:
1. Pastikan runtime memakai **T4 GPU** (*Runtime → Change runtime type → T4 GPU*).
2. Jika ingin menyimpan aman ke Google Drive, jalankan **Sel 1 (Mount Drive)** lalu **Sel 2 (Setup)**.
3. Render otomatis memakai mode cepat **`headless-shell` + shared memory + GPU `h264_nvenc`** (karena video ini komposisi 2D, tidak perlu `chrome-for-testing` + `angle-egl` yang lambat).
4. Jika merender **`SejarahIslam`** di **akun Colab lain**, cukup buka notebook ini di akun ke-2 lalu jalankan **Sel 1** (opsional) → **Sel 5** (Sel 5 sudah otomatis setup), lalu jalankan sel **Download MP4** di bawahnya.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/remotion-out

In [ ]:
%%shell
set -e

echo "=== 1. Cek GPU ==="
if command -v nvidia-smi >/dev/null 2>&1 && nvidia-smi >/dev/null 2>&1; then
  nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
else
  echo "WARNING: GPU NVIDIA tidak aktif! Disarankan pilih Runtime -> Change runtime type -> T4 GPU."
fi

echo "=== 2. Install dependensi Chromium & Font Linux ==="
DEBIAN_FRONTEND=noninteractive apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq \
  libnss3 libnspr4 libdbus-1-3 libatk1.0-0 libatk-bridge2.0-0 libcups2 \
  libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 \
  libxrandr2 libgbm1 libpango-1.0-0 libcairo2 libxshmfence1 \
  libvulkan1 fonts-liberation
(DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2t64 || DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2)

echo "=== 3. Install Node.js 20 ==="
NODE_MAJOR=$(node -v 2>/dev/null | sed 's/v\([0-9]*\).*/\1/' || echo 0)
if [ "$NODE_MAJOR" -lt 20 ]; then
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
  DEBIAN_FRONTEND=noninteractive apt-get install -y -qq nodejs
fi
node -v && npm -v

echo "=== 4. Clone / Update Repo & npm ci ==="
if [ -d /content/vidio/.git ]; then
  cd /content/vidio && git pull --ff-only
else
  rm -rf /content/vidio
  git clone --depth 1 https://github.com/giestar-id/remotion-vidio.git /content/vidio
fi
cd /content/vidio && npm ci

echo "Setup selesai! Siap render."

In [ ]:
# (Opsional) Tes cepat 10 detik pertama (frames 0-299) dengan 1 baris progress anti-lag
import importlib, os, subprocess, sys

if os.path.isdir('/content/vidio/.git'):
    subprocess.run(['git', '-C', '/content/vidio', 'pull', '--ff-only'], check=False)

out_dir = '/content/drive/MyDrive/remotion-out' if os.path.isdir('/content/drive/MyDrive') else '/content/remotion-out'
sys.path.insert(0, '/content/vidio/scripts')
import colab_render
importlib.reload(colab_render)
colab_render.run_render(
    'SejarahYunani',
    f'{out_dir}/test-yunani-10s.mp4',
    extra_args=['--frames=0-299'],
    cwd='/content/vidio',
)

In [ ]:
# Sel 4 - Render Video 1: SejarahYunani (1920x1080 @ 30fps, 24.131 frames) - 1 baris loading anti-lag
import importlib, os, subprocess, sys

if os.path.isdir('/content/vidio/.git'):
    subprocess.run(['git', '-C', '/content/vidio', 'pull', '--ff-only'], check=False)

out_dir = '/content/drive/MyDrive/remotion-out' if os.path.isdir('/content/drive/MyDrive') else '/content/remotion-out'
sys.path.insert(0, '/content/vidio/scripts')
import colab_render
importlib.reload(colab_render)
colab_render.run_render('SejarahYunani', f'{out_dir}/sejarah-yunani-1080p.mp4', cwd='/content/vidio')

In [ ]:
# (Opsional) Download langsung sejarah-yunani-1080p.mp4 ke laptop
from google.colab import files
import os

for p in [
    '/content/drive/MyDrive/remotion-out/sejarah-yunani-1080p.mp4',
    '/content/remotion-out/sejarah-yunani-1080p.mp4',
]:
    if os.path.exists(p):
        print('Mengunduh:', p)
        files.download(p)
        break
else:
    print('File sejarah-yunani-1080p.mp4 belum ditemukan. Jalankan sel render SejarahYunani dulu.')

In [ ]:
# Sel 5 - Render Video 2: SejarahIslam (1920x1080 @ 30fps, 27.280 frames) - 1 baris loading anti-lag
# Mandiri: jika dijalankan di akun Colab baru tanpa lewat Sel 2, setup akan dijalankan otomatis.
import importlib, os, subprocess, sys

if not os.path.isdir('/content/vidio/node_modules'):
    print('=== Auto-setup environment untuk akun baru ===', flush=True)
    setup_sh = r"""
    set -e
    DEBIAN_FRONTEND=noninteractive apt-get update -qq
    DEBIAN_FRONTEND=noninteractive apt-get install -y -qq \
      libnss3 libnspr4 libdbus-1-3 libatk1.0-0 libatk-bridge2.0-0 libcups2 \
      libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 \
      libxrandr2 libgbm1 libpango-1.0-0 libcairo2 libxshmfence1 \
      libvulkan1 fonts-liberation
    (DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2t64 || DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2)
    NODE_MAJOR=$(node -v 2>/dev/null | sed 's/v\([0-9]*\).*/\1/' || echo 0)
    if [ "$NODE_MAJOR" -lt 20 ]; then
      curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
      DEBIAN_FRONTEND=noninteractive apt-get install -y -qq nodejs
    fi
    rm -rf /content/vidio
    git clone --depth 1 https://github.com/giestar-id/remotion-vidio.git /content/vidio
    cd /content/vidio && npm ci --no-audit --no-fund
    """
    subprocess.run(['bash', '-c', setup_sh], check=True)
else:
    subprocess.run(['git', '-C', '/content/vidio', 'pull', '--ff-only'], check=False)

out_dir = '/content/drive/MyDrive/remotion-out' if os.path.isdir('/content/drive/MyDrive') else '/content/remotion-out'
sys.path.insert(0, '/content/vidio/scripts')
import colab_render
importlib.reload(colab_render)
colab_render.run_render('SejarahIslam', f'{out_dir}/sejarah-islam-1080p.mp4', cwd='/content/vidio')

In [ ]:
# Download langsung sejarah-islam-1080p.mp4 ke laptop
from google.colab import files
import os

for p in [
    '/content/drive/MyDrive/remotion-out/sejarah-islam-1080p.mp4',
    '/content/remotion-out/sejarah-islam-1080p.mp4',
]:
    if os.path.exists(p):
        print('Mengunduh:', p)
        files.download(p)
        break
else:
    print('File sejarah-islam-1080p.mp4 belum ditemukan. Jalankan sel render SejarahIslam dulu.')